In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.9.4 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [2]:
!uv pip install -r requirements.txt

Using Python 3.12.12 environment at: /usr
Resolved 186 packages in 1.48s
Prepared 63 packages in 1.84s
Uninstalled 5 packages in 18ms
Installed 63 packages in 84ms
 + backoff==2.2.1
 + coloredlogs==15.0.1
 + cyclopts==4.0.0
 + dataclasses-json==0.6.7
 + dnspython==2.8.0
 + email-validator==2.3.0
 + emoji==2.15.0
 + exceptiongroup==1.3.0
 + fastembed==0.7.3
 + fastmcp==2.12.5
 + filetype==1.2.0
 + groq==0.32.0
 + gunicorn==23.0.0
 + humanfriendly==10.0
 + isodate==0.7.2
 + jsonschema-path==0.3.4
 + langchain-cerebras==0.5.0
 + langchain-community==0.3.31
 + langchain-groq==0.3.8
 + langchain-nvidia-ai-endpoints==0.3.18
 + langchain-openai==0.3.35
 + langchain-qdrant==0.2.1
 + langdetect==1.0.9
 + langgraph==1.0.0
 + langgraph-checkpoint==2.1.2
 + langgraph-prebuilt==1.0.0
 + langgraph-sdk==0.2.9
 + lazy-object-proxy==1.12.0
 + loguru==0.7.3
 + marshmallow==3.26.1
 - mcp==1.17.0
 + mcp==1.16.0
 + mmh3==5.2.0
 + mypy-extensions==1.1.0
 + neo4j==6.0.2
 + olefile==0.47
 + onnxruntime==1.23.

In [3]:
!uv pip install langchain_text_splitters
!uv pip install langchain_community

Using Python 3.12.12 environment at: /usr
Audited 1 package in 86ms
Using Python 3.12.12 environment at: /usr
Audited 1 package in 85ms


In [4]:
!uv pip install qdrant-client[fastembed]
!uv pip install -qU langchain-qdrant

Using Python 3.12.12 environment at: /usr
Audited 1 package in 92ms


In [11]:
!pip install sentence_transformers

# Embedding & Neo4J

I would need API keys of Neo4j and Nvidia-AI, and then check embedding and retrieval for multiple Embedding Models

## ${\color{red}{NVIDIA-Models-not-accessible-due-to-geographical-unavailability}}$

Will have to go for hugging-face embeddings (embeddinggemma-300m)

In [8]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_document = """
# Features
## 4.2 Plant Identification / Disease Identification by Capturing Photo\n

### 4.2.1 Description and Priority\n
**Priority:** High

This feature allows users to identify plants by taking a photo using their smartphone.
It uses image recognition technology to compare the photo with its plant database, providing accurate plant identification and detailed information.

Additionally, the app can detect diseases in plants by analyzing photos of affected areas.
It offers recommendations for treatment and prevention.
Users can share identified plants and diseases on the in-app community forum or social media.
It creates a collaborative environment for plant enthusiasts to engage and seek advice.
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

[Document(metadata={'Header 1': 'Features', 'Header 2': '4.2 Plant Identification / Disease Identification by Capturing Photo', 'Header 3': '4.2.1 Description and Priority'}, page_content='# Features  \n## 4.2 Plant Identification / Disease Identification by Capturing Photo  \n### 4.2.1 Description and Priority  \n**Priority:** High  \nThis feature allows users to identify plants by taking a photo using their smartphone.\nIt uses image recognition technology to compare the photo with its plant database, providing accurate plant identification and detailed information.  \nAdditionally, the app can detect diseases in plants by analyzing photos of affected areas.\nIt offers recommendations for treatment and prevention.\nUsers can share identified plants and diseases on the in-app community forum or social media.\nIt creates a collaborative environment for plant enthusiasts to engage and seek advice.')]

In [9]:
for i in md_header_splits:
  print(i)

page_content='# Features  
## 4.2 Plant Identification / Disease Identification by Capturing Photo  
### 4.2.1 Description and Priority  
**Priority:** High  
This feature allows users to identify plants by taking a photo using their smartphone.
It uses image recognition technology to compare the photo with its plant database, providing accurate plant identification and detailed information.  
Additionally, the app can detect diseases in plants by analyzing photos of affected areas.
It offers recommendations for treatment and prevention.
Users can share identified plants and diseases on the in-app community forum or social media.
It creates a collaborative environment for plant enthusiasts to engage and seek advice.' metadata={'Header 1': 'Features', 'Header 2': '4.2 Plant Identification / Disease Identification by Capturing Photo', 'Header 3': '4.2.1 Description and Priority'}


# **Now Embedding & Inserting in Vector DB**

In [10]:
from google.colab import userdata

QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
QDRANT_URL = userdata.get("QDRANT_URL")
HUGGINGFACEHUB_API_TOKEN = userdata.get("HF_TOKEN")

In [11]:
!uv pip install huggingface_hub

Using Python 3.12.12 environment at: /usr
Audited 1 package in 87ms


In [12]:
!pip install -U numpy==1.26.4 scipy==1.12.0 scikit-learn==1.5.2
!pip install -U sentence-transformers

In [13]:
!uv pip install langchain_huggingface

Using Python 3.12.12 environment at: /usr
Audited 1 package in 87ms


In [14]:
from sentence_transformers import SentenceTransformer
# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="google/embeddinggemma-300m")
# embeddings2 = HuggingFaceHubEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2', huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN)

In [15]:
from langchain_qdrant import Qdrant, QdrantVectorStore
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings2 = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

doc_store = QdrantVectorStore.from_documents(
    documents=md_header_splits,
    embedding=embeddings2,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="test-cluster"
)